# Hiring Funnel Analysis

## Project Overview
Analyze recruiting pipeline data to understand drop-off rates, hiring speed, and conversion by role, source, and location.

## Learning Objectives
- Build and analyze a recruiting funnel from application to hire
- Calculate stage-wise conversion rates and bottleneck identification
- Compare hiring efficiency across sources, roles, and locations
- Analyze time-to-hire distributions and their drivers

## Problem Statement
The talent acquisition team needs to understand where candidates are dropping out of the hiring pipeline, which sources yield the best hires, and how long it takes to close positions across different roles.

## Why This Project Matters
A slow or leaky hiring funnel costs organizations qualified candidates, increases the time-to-productivity, and wastes recruiter effort. Funnel analytics enable targeted improvements.

## Dataset Overview
Synthetic recruiting dataset: ~5,000 applications across multiple roles, sources, and locations with stage progression data.

## Dataset Source and License Notes
- Synthetic data
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 5000
roles = np.random.choice(['Software Engineer', 'Data Analyst', 'Product Manager', 'Designer', 'Sales Rep', 'Operations'],
                           n, p=[0.25, 0.15, 0.12, 0.10, 0.22, 0.16])
sources = np.random.choice(['LinkedIn', 'Referral', 'Job Board', 'Company Website', 'Recruiter', 'Campus'],
                             n, p=[0.28, 0.18, 0.20, 0.15, 0.12, 0.07])
locations = np.random.choice(['New York', 'San Francisco', 'Austin', 'Chicago', 'Remote'], n, p=[0.22, 0.20, 0.18, 0.15, 0.25])

apply_date = pd.Timestamp('2023-01-01') + pd.to_timedelta(np.random.randint(0, 365, n), unit='D')

# Stage progression with realistic drop-offs
screen_pass = np.random.random(n) < np.where(sources == 'Referral', 0.65, 0.45)
phone_pass = screen_pass & (np.random.random(n) < 0.55)
onsite_pass = phone_pass & (np.random.random(n) < 0.45)
offer_made = onsite_pass & (np.random.random(n) < 0.70)
hired = offer_made & (np.random.random(n) < 0.80)

# Days between stages
days_to_screen = np.random.randint(1, 8, n)
days_to_phone = np.random.randint(3, 14, n)
days_to_onsite = np.random.randint(5, 21, n)
days_to_offer = np.random.randint(2, 10, n)
days_to_accept = np.random.randint(1, 14, n)

total_days = days_to_screen + days_to_phone * phone_pass + days_to_onsite * onsite_pass + days_to_offer * offer_made + days_to_accept * hired

def final_stage(row):
    if row['Hired']: return 'Hired'
    if row['OfferMade']: return 'Offer Declined'
    if row['OnsitePass']: return 'Post-Onsite Reject'
    if row['PhonePass']: return 'Post-Phone Reject'
    if row['ScreenPass']: return 'Post-Screen Reject'
    return 'Screen Reject'

df = pd.DataFrame({
    'ApplicationID': [f'APP{i:05d}' for i in range(n)],
    'Role': roles, 'Source': sources, 'Location': locations,
    'ApplyDate': apply_date,
    'ScreenPass': screen_pass, 'PhonePass': phone_pass,
    'OnsitePass': onsite_pass, 'OfferMade': offer_made, 'Hired': hired,
    'TotalDays': total_days,
})
df['FinalStage'] = df.apply(final_stage, axis=1)

print(f'Dataset: {df.shape}')
print(f'Hired: {df["Hired"].sum()} ({df["Hired"].mean():.1%})')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Applications: {df["ApplicationID"].nunique()}')
print(f'\nFinal stage distribution:')
print(df['FinalStage'].value_counts())

## Overall Funnel Analysis

In [ ]:
stages = ['Applied', 'Screen Pass', 'Phone Pass', 'Onsite Pass', 'Offer Made', 'Hired']
counts = [len(df), df['ScreenPass'].sum(), df['PhonePass'].sum(),
          df['OnsitePass'].sum(), df['OfferMade'].sum(), df['Hired'].sum()]

funnel = pd.DataFrame({'Stage': stages, 'Count': counts})
funnel['Conversion'] = [1.0] + [counts[i]/counts[i-1] if counts[i-1] > 0 else 0 for i in range(1, len(counts))]
funnel['CumulConv'] = [c/counts[0] for c in counts]
print('Overall Hiring Funnel:')
print(funnel.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(stages[::-1], [c for c in counts[::-1]], edgecolor='black', color='steelblue')
for bar, c in zip(bars, counts[::-1]):
    ax.text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2, f'{c}', va='center')
ax.set_title('Hiring Funnel — Overall')
ax.set_xlabel('Candidates')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'overall_funnel.png'), dpi=100, bbox_inches='tight')
plt.show()

## Conversion Rates by Source

In [ ]:
src_funnel = df.groupby('Source').agg(
    applied=('ApplicationID', 'count'),
    screen=('ScreenPass', 'sum'), phone=('PhonePass', 'sum'),
    onsite=('OnsitePass', 'sum'), offer=('OfferMade', 'sum'), hired=('Hired', 'sum'),
).astype(int)
src_funnel['hire_rate'] = (src_funnel['hired'] / src_funnel['applied']).round(3)
src_funnel['screen_rate'] = (src_funnel['screen'] / src_funnel['applied']).round(3)
print('Funnel by Source:')
print(src_funnel.sort_values('hire_rate', ascending=False))

fig, ax = plt.subplots(figsize=(10, 5))
src_funnel.sort_values('hire_rate')['hire_rate'].plot.barh(ax=ax, edgecolor='black', color='coral')
ax.set_title('Overall Hire Rate by Source')
ax.set_xlabel('Hire Rate')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'source_hire_rate.png'), dpi=100, bbox_inches='tight')
plt.show()

## Conversion Rates by Role

In [ ]:
role_funnel = df.groupby('Role').agg(
    applied=('ApplicationID', 'count'),
    hired=('Hired', 'sum'),
    avg_days=('TotalDays', 'mean'),
).astype({'applied': int, 'hired': int})
role_funnel['hire_rate'] = (role_funnel['hired'] / role_funnel['applied']).round(3)
print('Hiring by Role:')
print(role_funnel.sort_values('hire_rate', ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
role_funnel.sort_values('hire_rate')['hire_rate'].plot.barh(ax=axes[0], edgecolor='black', color='teal')
axes[0].set_title('Hire Rate by Role')
role_funnel.sort_values('avg_days')['avg_days'].plot.barh(ax=axes[1], edgecolor='black', color='goldenrod')
axes[1].set_title('Avg Days in Pipeline by Role')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'role_analysis.png'), dpi=100, bbox_inches='tight')
plt.show()

## Time-to-Hire Distribution

In [ ]:
hired_df = df[df['Hired']]
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(hired_df['TotalDays'], bins=30, edgecolor='black', color='steelblue', alpha=0.7)
ax.axvline(hired_df['TotalDays'].median(), color='red', linestyle='--', label=f'Median: {hired_df["TotalDays"].median():.0f} days')
ax.set_title('Time-to-Hire Distribution (Hired Candidates)')
ax.set_xlabel('Days')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'time_to_hire.png'), dpi=100, bbox_inches='tight')
plt.show()
print(f'Time-to-hire stats:\n{hired_df["TotalDays"].describe().round(1)}')

## Monthly Application Volume

In [ ]:
df['Month'] = df['ApplyDate'].dt.to_period('M')
monthly = df.groupby('Month').agg(applications=('ApplicationID', 'count'), hired=('Hired', 'sum'))
monthly['hire_rate'] = (monthly['hired'] / monthly['applications']).round(3)
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(monthly.index.astype(str), monthly['applications'], edgecolor='black', color='steelblue', alpha=0.7, label='Applications')
ax2 = ax.twinx()
ax2.plot(monthly.index.astype(str), monthly['hire_rate'], color='red', marker='o', label='Hire Rate')
ax.set_title('Monthly Applications & Hire Rate')
ax.set_xlabel('Month')
ax.set_ylabel('Applications')
ax2.set_ylabel('Hire Rate')
ax.tick_params(axis='x', rotation=45)
ax.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'monthly_volume.png'), dpi=100, bbox_inches='tight')
plt.show()

## Location Analysis

In [ ]:
loc_stats = df.groupby('Location').agg(
    applications=('ApplicationID', 'count'),
    hired=('Hired', 'sum'),
    avg_days=('TotalDays', 'mean'),
).round(1)
loc_stats['hire_rate'] = (loc_stats['hired'] / loc_stats['applications']).round(3)
print('Hiring by Location:')
print(loc_stats.sort_values('hire_rate', ascending=False))

## Interpretation and Business Insight
- **Referrals** have the highest funnel conversion — they pass screening at higher rates and ultimately convert better
- **Screen reject** is the largest single drop-off point — tighter job description targeting could reduce wasted volume
- **Onsite-to-offer** is a significant bottleneck — interview calibration and candidate experience investments matter here
- **Time-to-hire** varies by role, with some roles consistently slower — these need dedicated sourcing pipelines
- **Offer decline rates** signal competitiveness of compensation packages

## Limitations
- Synthetic data with simplified stage logic
- No candidate quality scores or interviewer feedback
- No cost-per-hire or recruiter workload data
- Assumes linear funnel — real pipelines have loops and re-engagements
- No diversity or DEI dimension analysis

## How to Improve This Project
- Add cost-per-hire by source and role
- Include candidate satisfaction / NPS scores
- Track re-application and talent pool re-engagement
- Add DEI funnel analysis
- Model optimal recruiter capacity and workload

## Production Considerations
- Weekly automated funnel reports for TA leadership
- Real-time dashboards connected to ATS (Greenhouse, Lever)
- Alerts for bottleneck stage degradation
- Source ROI tracking for recruiting budget allocation

## Common Mistakes
- Measuring only the hire rate without understanding where candidates drop off
- Not segmenting by role — aggregate numbers hide role-specific problems
- Ignoring time-in-stage: a 100% conversion that takes 60 days loses candidates
- Treating all sources equally in budget allocation despite different conversion rates

## Mini Challenge / Exercises
1. Which source has the highest cost-efficiency if LinkedIn costs $500/hire, Referral $1000 bonus, and Job Boards $300/posting?
2. Calculate the stage with the largest absolute candidate loss.
3. If you improved screen-to-phone conversion by 10%, how many additional hires would you expect?
4. Which role × source combination has the best hire rate?

## Final Summary / Key Takeaways
- Hiring funnel analytics reveal where candidates are lost and which channels perform best
- Referral programs typically have the best conversion rates and should be expanded
- Bottleneck identification (screen, onsite, offer) enables targeted process improvements
- Time-to-hire tracking prevents losing candidates to competitor offers
- Source-level ROI analysis drives smarter recruiting investment